In [1]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parents[1])
sys.path.insert(0, parent_dir)

In [2]:
from src.paths import DATA_DIR
from src.Qlassifier.results import Results
from src.Qlassifier.prediction import InstructPredictor
from notebooks.modelling import scope

/home/ykip10/projects/Qlassifier/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Run the model
predictors: list[InstructPredictor] = []
results: list[Results] = []
for subject, exam in zip(scope.SUBJECTS, scope.EXAMS):
    exam_path = DATA_DIR / subject / "past_exams" / f"{exam}.pdf"
    sd_path = DATA_DIR / subject / "study_design" / f"{subject}_sd.docx"
    model = InstructPredictor(sd_path, subject)
    res = model.run(exam_path)
    predictors.append(model)
    results.append(res)

No sentence-transformers model found with name hkunlp/instructor-large. Creating a new one with mean pooling.
/home/ykip10/projects/Qlassifier/src/preprocessor/loading.py:129: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~(df["label"].str.contains(
`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.
`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.
No sentence-transformers model found with name hkunlp/instructor-large. Creating a new one with mean pooling.
`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.
`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.


## Chemistry

In [4]:
true_topic_mcq = [
    0, 2, 6, 5, 10, 9, 0, 4, 0, 4,
    3, 3, 6, 4, 0, 9, 1, 9, 6, 8,
    10 ,2, 6, 2, 5, 11, 12, 2, 9, 2,
]

true_topic_indices = true_topic_mcq + [
    6, 7, 7, 6, 6, 2, 2, 2, 2,   # Q1
    6, 10, 10, 10, 6, 10, 6, 10, # Q2
    0, 0, 0, 0, 0, 6,            # Q3
    7, 6, 7, 7, 7,               # Q4
    0, 4, 4, 3,                  # Q5
    2, 2, 2, 2, 2,               # Q6 
    9, 9, 9, 9, 9,               # Q7
    3, 11, 11, 1, 1, 12, 11, 12, # Q8
    5, 5                         # Q9
]

chem_res = results[0]
chem_res.correct_topics = true_topic_indices
round(chem_res.summary(by="overall"), 2)

Accuracy              0.48
Top 3 Accuracy        0.80
Macro-Precision       0.44
Macro-Recall          0.39
Macro-F1-Score        0.36
Weighted-Precision    0.59
Weighted-Recall       0.48
Weighted-F1-Score     0.48
dtype: float64

In [5]:
chem_res.pred_df

,label,text,pred_topic,pred_topic_idx,confidence,true_topic_idx,true_topic
0,Question 1,Which one of the following correctly represen...,Carbon-based fuels,0,0.435189,0,Carbon-based fuels
1,Question 2,Which one of the following statements is corr...,Primary galvanic cells and fuel cells as sourc...,2,0.895996,2,Primary galvanic cells and fuel cells as sourc...
2,Question 3,"Molecules X, Y and Z all have the same number...","Structure, nomenclature and properties of orga...",6,0.424464,6,"Structure, nomenclature and properties of orga..."
3,Question 4,A cell can be considered a secondary cell if ...,Primary galvanic cells and fuel cells as sourc...,2,0.617723,5,Production of chemicals using electrolysis
4,Question 5,Consider the following statements about coenz...,Rates of chemical reactions,3,0.438562,10,Medicinal chemistry
...,...,...,...,...,...,...,...
77,Question 8e.ii.,A student performed an experiment to compare ...,Measuring changes in chemical reactions,1,0.421875,12,Scientific evidence
78,Question 8f.,A student performed an experiment to compare ...,Measuring changes in chemical reactions,1,0.280827,11,Investigation design
79,Question 8g.,A student performed an experiment to compare ...,Measuring changes in chemical reactions,1,0.292095,12,Scientific evidence
80,Question 9a.,Purified copper contains at least 99.8% coppe...,Production of chemicals using electrolysis,5,1.000000,5,Production of chemicals using electrolysis


In [6]:
round(chem_res.summary("topic"), 2)

label,Carbon-based fuels,Measuring changes in chemical reactions,Primary galvanic cells and fuel cells as sources of energy,Rates of chemical reactions,Extent of chemical reactions,Production of chemicals using electrolysis,"Structure, nomenclature and properties of organic compounds",Reactions of organic compounds,Laboratory analysis of organic compounds,Instrumental analysis of organic compounds,Medicinal chemistry,Investigation design,Scientific evidence,Science communication
Predicted Topic Counts,8.00,17.00,7.00,2.00,4.00,6.00,13.00,12.00,0.00,9.00,1.00,3.00,0.00,0.0
Predicted Topic Proportions,0.10,0.21,0.09,0.02,0.05,0.07,0.16,0.15,0.00,0.11,0.01,0.04,0.00,0.0
Topic Count,10.00,3.00,14.00,4.00,5.00,4.00,12.00,6.00,1.00,9.00,7.00,4.00,3.00,0.0
Topic Proportion,0.12,0.04,0.17,0.05,0.06,0.05,0.15,0.07,0.01,0.11,0.09,0.05,0.04,0.0
Precision,0.50,0.12,0.71,0.50,0.75,0.33,0.54,0.33,0.00,1.00,1.00,0.33,0.00,0.0
Recall,0.40,0.67,0.36,0.25,0.60,0.50,0.58,0.67,0.00,1.00,0.14,0.25,0.00,0.0
F1-Score,0.44,0.20,0.48,0.33,0.67,0.40,0.56,0.44,0.00,1.00,0.25,0.29,0.00,0.0


This model has the same issues as the sentence-transformer one. It is heavily overestimating the "Measuring changes in Chemical reactions" and "Reactions of Organic Compounds" fields and under-representing medicinal chemistry. The TF-IDF approach is generally the best of the three for this subject. 

## Specialist Math

In [7]:
mcq_labels = [
    0, 1, 1, 2, 2, 4, 4, 4, 7, 3, 
    3, 5, 5, 6, 6, 8, 6, 7, 9, 11,
]

ground_truth = mcq_labels + [
    1, 3, 3, 3, 7, 8, 3, 3,            # Q1
    2, 2, 2, 2, 2, 2, 2, 2,            # Q2
    3, 3, 3, 3, 3, 3,                  # Q3
    4, 4, 4, 4, 4, 4, 1, 4,            # Q4
    6, 6, 7, 7, 7, 7,                  # Q5
    11, 11, 11, 12, 12, 12, 12, 12, 10 # Q6  

]

math_res = results[1]
math_res.correct_topics = ground_truth
round(math_res.summary(by="overall"), 2)

Accuracy              0.75
Top 3 Accuracy        0.97
Macro-Precision       0.67
Macro-Recall          0.71
Macro-F1-Score        0.65
Weighted-Precision    0.81
Weighted-Recall       0.75
Weighted-F1-Score     0.76
dtype: float64

In [8]:
round(math_res.summary(by="topic"), 2)

label,Discrete mathematics: Logic and proof,"Functions, relations and graphs","Algebra, number and structure: Complex numbers",Calculus: Differential calculus and integral calculus,Calculus: Differential equations,Calculus: Kinematics: rectilinear motion,Space and measurement: Vectors,Space and measurement: Vector and Cartesian equations,Space and measurement: Vector calculus,"Data analysis, probability and statistics: Distribution of linear combinations of random variables","Data analysis, probability and statistics: Distribution of the sample mean","Data analysis, probability and statistics: Confidence intervals for the population mean","Data analysis, probability and statistics: Hypothesis testing for a population mean with a sample drawn from a normal distribution of known variance, or for a large sample"
Predicted Topic Counts,2.00,3.00,10.00,10.00,12.00,2.00,5.00,5.00,5.00,0.00,4.00,2.00,5.00
Predicted Topic Proportions,0.03,0.05,0.15,0.15,0.18,0.03,0.08,0.08,0.08,0.00,0.06,0.03,0.08
Topic Count,1.00,4.00,10.00,13.00,10.00,2.00,5.00,7.00,2.00,1.00,1.00,4.00,5.00
Topic Proportion,0.02,0.06,0.15,0.20,0.15,0.03,0.08,0.11,0.03,0.02,0.02,0.06,0.08
Precision,0.50,0.67,1.00,0.90,0.75,1.00,0.80,0.80,0.20,0.00,0.25,1.00,0.80
Recall,1.00,0.50,1.00,0.69,0.90,1.00,0.80,0.57,0.50,0.00,1.00,0.50,0.80
F1-Score,0.67,0.57,1.00,0.78,0.82,1.00,0.80,0.67,0.29,0.00,0.40,0.67,0.80


This is by far the best spceialist math model. It seems a bit counterintuitive that the instructor performs so much better on specialist math, in which we've removed almost all the LaTeX, compared to chemistry which is almost purely text. There seems to be a real problem with the way the model is tuned; it is just bad at understanding domain-specific chemistry semantics. 

In [9]:
Path().resolve()

PosixPath('/home/ykip10/projects/Qlassifier/notebooks/modelling')

In [10]:
math_predictor = predictors[1]

In [11]:
import pickle

Path("saved_results").mkdir(exist_ok=True)

with open("saved_results/instruct_chem_res.pkl", "wb") as f:
    pickle.dump(chem_res, f)
with open("saved_results/instruct_math_res.pkl", "wb") as f:
    pickle.dump(math_res, f)